# Healthcare Insurance Claims – Data Warehouse Analytics
## Day 3: Python Analytics & Visualization

**Project:** Midterm Capstone – 3-Day Sprint  
**Focus:** Analyze the masked data warehouse using Pandas and Matplotlib

---
### What this notebook covers
1. Connect to the warehouse
2. Total claims summary
3. Monthly paid amount trend
4. Top procedures by utilisation
5. Provider performance scorecard
6. Denial rate breakdown
7. Length of stay distribution
8. Member utilisation segments
9. Geographic breakdown by state
10. Payment method breakdown
11. All Matplotlib charts

> **Run step4_load_full.py first before opening this notebook.**

---
## Setup – Connect to the Warehouse

In [ ]:
import sys
from pathlib import Path

# Add the capstone root to Python path so we can import config
capstone_root = Path(".").resolve().parent
sys.path.insert(0, str(capstone_root))

import sqlite3
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from config import DB_PATH, CHARTS_DIR

# Connect to the warehouse
conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH}")

# Helper: run SQL and return a DataFrame
def query(sql):
    return pd.read_sql_query(sql, conn)

# Helper: save chart to file
def save(fig, name):
    path = CHARTS_DIR / name
    fig.savefig(path, dpi=120, bbox_inches="tight")
    print(f"Saved: {path.name}")

---
## Q1 – Total Claims Summary

In [ ]:
df_summary = query("""
    SELECT
        COUNT(*)                        AS total_claims,
        ROUND(SUM(allowed_amount), 2)   AS total_allowed,
        ROUND(SUM(paid_amount), 2)      AS total_paid,
        ROUND(AVG(paid_amount), 2)      AS avg_paid_per_claim,
        SUM(is_denied)                  AS total_denied,
        ROUND(100.0 * SUM(is_denied) / COUNT(*), 2) AS denial_rate_pct
    FROM fact_claim
""")
df_summary

---
## Q2 – Monthly Paid Amount Trend

In [ ]:
df_monthly = query("""
    SELECT d.year, d.month, d.month_name,
           COUNT(fc.claim_number) AS claims,
           ROUND(SUM(fc.paid_amount)/1000000, 2) AS paid_millions
    FROM fact_claim fc
    JOIN dim_date d ON fc.admit_date_key = d.date_key
    GROUP BY d.year, d.month, d.month_name
    ORDER BY d.year, d.month
""")
df_monthly["label"] = df_monthly["month_name"].str[:3] + " " + df_monthly["year"].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(df_monthly["label"], df_monthly["paid_millions"], color="#4C72B0", alpha=0.85)
ax.set_title("Monthly Total Paid Amount", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Paid Amount (Millions ₹)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₹{x:.1f}M"))
plt.xticks(rotation=60, ha="right", fontsize=7)
plt.tight_layout()
save(fig, "01_monthly_paid.png")
plt.show()

---
## Q3 – Top 10 Procedures by Utilisation

In [ ]:
df_proc = query("""
    SELECT procedure_code,
           COUNT(*) AS times_used,
           SUM(units) AS total_units,
           ROUND(SUM(line_amount), 2) AS total_billed
    FROM fact_claim_line
    GROUP BY procedure_code
    ORDER BY times_used DESC
    LIMIT 10
""")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_proc["procedure_code"], df_proc["times_used"], color="#55A868", alpha=0.85)
ax.set_title("Top 10 Procedures by Utilisation", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Claim Lines")
ax.invert_yaxis()
plt.tight_layout()
save(fig, "02_top_procedures.png")
plt.show()
df_proc

---
## Q4 – Provider Performance Scorecard

In [ ]:
df_prov = query("""
    SELECT dp.provider_id, dp.specialty, dp.state,
           COUNT(fc.claim_number) AS total_claims,
           ROUND(SUM(fc.paid_amount)/1000, 1) AS paid_thousands,
           ROUND(AVG(fc.paid_amount), 2) AS avg_paid,
           ROUND(AVG(fc.length_of_stay), 1) AS avg_los,
           ROUND(100.0 * SUM(fc.is_denied) / COUNT(*), 1) AS denial_rate_pct
    FROM fact_claim fc
    JOIN dim_provider dp ON fc.provider_id = dp.provider_id
    GROUP BY dp.provider_id, dp.specialty, dp.state
    ORDER BY paid_thousands DESC
    LIMIT 15
""")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(df_prov["provider_id"], df_prov["paid_thousands"], color="#C44E52", alpha=0.85)
axes[0].set_title("Top 15 Providers – Total Paid", fontsize=12)
axes[0].set_xlabel("Total Paid (Thousands ₹)")
axes[0].invert_yaxis()
axes[0].tick_params(axis="y", labelsize=7)

colours = ["#C44E52" if r > 10 else "#55A868" for r in df_prov["denial_rate_pct"].fillna(0)]
axes[1].barh(df_prov["provider_id"], df_prov["denial_rate_pct"].fillna(0), color=colours, alpha=0.85)
axes[1].set_title("Top 15 Providers – Denial Rate %", fontsize=12)
axes[1].set_xlabel("Denial Rate (%)")
axes[1].invert_yaxis()
axes[1].tick_params(axis="y", labelsize=7)

fig.suptitle("Provider Performance", fontsize=14, fontweight="bold")
plt.tight_layout()
save(fig, "03_provider_performance.png")
plt.show()
df_prov

---
## Q5 – Denial Rate Breakdown

In [ ]:
df_denial = query("""
    SELECT
        CASE WHEN denial_code = '' OR denial_code IS NULL THEN 'NO DENIAL'
             ELSE denial_code END AS denial_reason,
        COUNT(*) AS claim_count,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM fact_claim), 2) AS pct_of_total
    FROM fact_claim
    GROUP BY denial_reason
    ORDER BY claim_count DESC
""")

fig, ax = plt.subplots(figsize=(8, 5))
colours = ["#4C72B0" if c == "NO DENIAL" else "#C44E52" for c in df_denial["denial_reason"]]
ax.bar(df_denial["denial_reason"].astype(str), df_denial["claim_count"], color=colours, alpha=0.85)
ax.set_title("Claims by Denial Code", fontsize=14, fontweight="bold")
ax.set_xlabel("Denial Code")
ax.set_ylabel("Number of Claims")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
save(fig, "04_denial_rate.png")
plt.show()
df_denial

---
## Q6 – Length of Stay Distribution

In [ ]:
df_los = query("""
    SELECT length_of_stay, COUNT(*) AS claim_count,
           ROUND(AVG(paid_amount), 2) AS avg_paid
    FROM fact_claim
    WHERE length_of_stay >= 0 AND length_of_stay <= 30
    GROUP BY length_of_stay
    ORDER BY length_of_stay
""")

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_los["length_of_stay"], df_los["claim_count"], color="#8172B2", alpha=0.85, width=0.8)
ax.set_title("Length of Stay Distribution (0–30 Days)", fontsize=14, fontweight="bold")
ax.set_xlabel("Days in Hospital")
ax.set_ylabel("Number of Claims")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
save(fig, "05_los_distribution.png")
plt.show()

---
## Q7 – Member Utilisation Segments

In [ ]:
df_mem = query("""
    SELECT dm.gender,
           COUNT(fc.claim_number) AS total_claims,
           ROUND(SUM(fc.paid_amount), 2) AS total_paid
    FROM fact_claim fc
    JOIN dim_member dm ON fc.member_id = dm.member_id
    GROUP BY dm.member_id, dm.gender
    ORDER BY total_paid DESC
    LIMIT 200
""")

fig, ax = plt.subplots(figsize=(10, 6))
for gender, colour, marker in [("male", "#4C72B0", "o"), ("female", "#DD8452", "^")]:
    sub = df_mem[df_mem["gender"] == gender]
    ax.scatter(sub["total_claims"], sub["total_paid"] / 1000,
               label=gender.capitalize(), color=colour, alpha=0.6, s=40, marker=marker)
ax.set_title("Member Utilisation: Claims vs Total Paid (Top 200)", fontsize=13, fontweight="bold")
ax.set_xlabel("Total Claims")
ax.set_ylabel("Total Paid (Thousands ₹)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₹{x:.0f}K"))
ax.legend()
plt.tight_layout()
save(fig, "06_member_utilisation.png")
plt.show()

---
## Q8 – Geographic Breakdown by State

In [ ]:
df_geo = query("""
    SELECT dm.state,
           COUNT(fc.claim_number) AS total_claims,
           ROUND(SUM(fc.paid_amount)/1000000, 2) AS paid_millions
    FROM fact_claim fc
    JOIN dim_member dm ON fc.member_id = dm.member_id
    GROUP BY dm.state
    ORDER BY paid_millions DESC
    LIMIT 15
""")

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df_geo["state"], df_geo["paid_millions"], color="#64B5CD", alpha=0.85)
ax.set_title("Total Paid Amount by State (Top 15)", fontsize=14, fontweight="bold")
ax.set_xlabel("State")
ax.set_ylabel("Total Paid (Millions ₹)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₹{x:.1f}M"))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
save(fig, "07_geographic.png")
plt.show()
df_geo

---
## Q9 – Payment Method Breakdown

In [ ]:
df_pay = query("""
    SELECT payment_method,
           COUNT(*) AS count,
           ROUND(SUM(paid_amount)/1000000, 2) AS paid_millions
    FROM fact_payment
    WHERE payment_method IS NOT NULL AND payment_method != 'nan'
    GROUP BY payment_method
    ORDER BY paid_millions DESC
""")

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(df_pay["paid_millions"],
       labels=df_pay["payment_method"],
       autopct="%1.1f%%", startangle=140,
       colors=["#4C72B0","#DD8452","#55A868","#C44E52"])
ax.set_title("Payment Method Breakdown (by Total Paid)", fontsize=13, fontweight="bold")
plt.tight_layout()
save(fig, "08_payment_methods.png")
plt.show()
df_pay

---
## Optimization Summary

### Indexes added (in DDL)
| Index | Table | Column | Purpose |
|---|---|---|---|
| idx_fc_member | fact_claim | member_id | Speed up member JOINs |
| idx_fc_provider | fact_claim | provider_id | Speed up provider JOINs |
| idx_fc_admit | fact_claim | admit_date_key | Speed up date range filters |
| idx_fc_denied | fact_claim | is_denied | Speed up denial rate queries |
| idx_fcl_claim | fact_claim_line | claim_number | Speed up line-level JOINs |
| idx_fp_claim | fact_payment | claim_number | Speed up payment JOINs |

### Vectorized Pandas operations used
- `pd.to_numeric(..., errors='coerce')` — converts whole columns at once instead of row-by-row
- `pd.to_datetime(..., errors='coerce')` — same for dates
- Boolean masks (`df[mask]`) instead of loops for filtering
- `Series.map(dict)` for FK remapping — much faster than `.apply(lambda)`

### Data Masking impact on analytics
- Masked IDs are deterministic — JOINs between fact_claim, dim_member, dim_provider all work correctly
- City and state kept in clear → geographic analysis is unaffected
- DOB replaced by birth_year → age band analysis still possible
- Procedure codes and diagnosis codes not masked → clinical analytics work normally

In [ ]:
conn.close()
print("Done. All charts saved to:", CHARTS_DIR)